# 13 — Ce que le notebook 12 cachait, et un problème où le deep hedging brille vraiment

## Le post-mortem du notebook 12

Le notebook 12 affichait un résultat spectaculaire : réseau 2 instruments **CVaR 2.45** contre delta-vega classique **6.54**. En creusant, deux choses honnêtes sont apparues, et elles sont plus intéressantes que le chiffre brut.

D'abord, en regardant *ce que le réseau tient au cours du temps*, on a vu qu'il achète environ 0.93 d'option de couverture au départ, la garde quasi constante, et **ne touche presque pas au sous-jacent** (turnover 0.06 contre 2.04 pour l'option). Autrement dit il a appris un **calendar spread quasi statique**, pas une couverture dynamique sophistiquée.

Ensuite, on a testé un hedge *purement statique* (acheter l'option une fois, ne plus rien faire) : il donne déjà **CVaR 3.90**. Et le delta-vega classique à 6.54 est en réalité **battu par le fait de ne rien faire d'autre que tenir une option**, parce qu'il rééquilibre les deux grecques à chaque pas et se saigne en coûts.

Ce notebook fait deux choses. La **brique A** établit l'échelle honnête du problème du notebook 12, pour voir d'où vient vraiment le gain. La **brique B** change de passif pour un cas où *rien de statique ne marche* et où le deep hedging domine réellement : une **option à barrière**.


## Brique A — l'échelle honnête du problème vanille

On reconstruit toutes les stratégies sur les mêmes trajectoires, du plus bête au plus malin. Setup identique au notebook 12 (on reconstruit les grilles de prix Heston, ~1 min).


In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.interpolate import RegularGridInterpolator
from scipy.stats import norm

S0, K, mu, r, T1 = 100., 100., 0.05, 0.02, 1.0
v0, kappa, theta, xi, rho = 0.04, 2.0, 0.04, 0.3, -0.7
Kh, T2 = 100., 2.0
n, cost, alpha = 63, 0.01, 0.95
dt = T1/n

def heston_cf(phi, S0, v0, r, kappa, theta, xi, rho, T):
    out = []
    for u, b in [(0.5, kappa - rho*xi), (-0.5, kappa)]:
        d = np.sqrt((rho*xi*1j*phi - b)**2 - xi**2*(2*u*1j*phi - phi**2))
        g = (b - rho*xi*1j*phi + d)/(b - rho*xi*1j*phi - d)
        C = r*1j*phi*T + (kappa*theta/xi**2)*((b - rho*xi*1j*phi + d)*T - 2*np.log((1-g*np.exp(d*T))/(1-g)))
        D = (b - rho*xi*1j*phi + d)/xi**2 * ((1-np.exp(d*T))/(1-g*np.exp(d*T)))
        out.append(np.exp(C + D*v0 + 1j*phi*np.log(S0)))
    return out
def heston_call(S0, v0, r, kappa, theta, xi, rho, T, K):
    def integ(phi, i):
        f = heston_cf(phi, S0, v0, r, kappa, theta, xi, rho, T)[i]
        return (np.exp(-1j*phi*np.log(K))*f/(1j*phi)).real
    P1 = 0.5 + quad(integ, 1e-8, 200, args=(0,), limit=200)[0]/np.pi
    P2 = 0.5 + quad(integ, 1e-8, 200, args=(1,), limit=200)[0]/np.pi
    return S0*P1 - K*np.exp(-r*T)*P2

def build_grid(Sg, vg, tg, Kstrike):
    G = np.zeros((len(Sg), len(vg), len(tg)))
    for i, S in enumerate(Sg):
        for j, v in enumerate(vg):
            for k, tau in enumerate(tg):
                G[i, j, k] = heston_call(S, v, r, kappa, theta, xi, rho, tau, Kstrike)
    return G

Sg = np.linspace(55, 175, 25); vg = np.linspace(0.005, 0.15, 12)
tgO = np.linspace(1.0, 2.0, 13); tgL = np.linspace(0.01, 1.0, 15)
t0 = time.time()
Ointerp = RegularGridInterpolator((Sg, vg, tgO), build_grid(Sg, vg, tgO, Kh), bounds_error=False, fill_value=None)
Pinterp = RegularGridInterpolator((Sg, vg, tgL), build_grid(Sg, vg, tgL, K),  bounds_error=False, fill_value=None)
print(f"grilles construites en {time.time()-t0:.0f} s")

premium = float(Pinterp([[S0, v0, T1]])[0]); O0 = float(Ointerp([[S0, v0, T2]])[0])

def sim_heston(m, drift, seed):
    rng = np.random.default_rng(seed)
    S = np.empty((m, n+1)); v = np.empty((m, n+1)); S[:,0] = S0; v[:,0] = v0
    for k in range(n):
        Z1 = rng.standard_normal(m); Z2 = rho*Z1 + np.sqrt(1-rho**2)*rng.standard_normal(m)
        vk = np.maximum(v[:,k], 0.0)
        v[:,k+1] = np.maximum(v[:,k] + kappa*(theta-vk)*dt + xi*np.sqrt(vk*dt)*Z2, 0.0)
        S[:,k+1] = S[:,k]*np.exp((drift - 0.5*vk)*dt + np.sqrt(vk*dt)*Z1)
    return S, v
def cvar(pnl, a=0.95):
    loss = -pnl; return loss[loss >= np.quantile(loss, a)].mean()

print(f"prime passif = {premium:.3f}   prix option couverture = {O0:.3f}")


### Les cinq stratégies, de la plus naïve à la plus fine

On price le passif (call $K=100$, $T_1=1$), on encaisse la prime, et on compare. `greeks` calcule $\partial/\partial S$ et $\partial/\partial v$ par différences finies sur les interpolateurs (comme au notebook 12).


In [ ]:
hS_, hv_ = 1.0, 0.005
def greeks(interp, Sk, vk, tau):
    tau = np.full_like(Sk, tau); vm = np.maximum(vk-hv_, 1e-4)
    dS = (interp(np.c_[Sk+hS_, vk, tau]) - interp(np.c_[Sk-hS_, vk, tau]))/(2*hS_)
    dv = (interp(np.c_[Sk, vk+hv_, tau]) - interp(np.c_[Sk, vm, tau]))/(vk+hv_-vm)
    return dS, dv

m = 80_000; S, v = sim_heston(m, mu, 7); times = np.linspace(0, T1, n+1)
Ofin = Ointerp(np.c_[S[:,-1], v[:,-1], np.full(m, T2-T1)]); payoff = np.maximum(S[:,-1]-K, 0.)

# 1) ne rien faire
cvar_none = cvar(premium*np.exp(r*T1) - payoff)

# 2) statique : sous-jacent seul, h actions tenues, h optimise
best = min((cvar((premium-h*S0-cost*h*S0)*np.exp(r*T1) + h*S[:,-1] - payoff), h) for h in np.linspace(0,1,21))
cvar_stat_S = best[0]

# 3) statique : 1 option tenue, h optimise
best = min((cvar((premium-h*O0-cost*h*O0)*np.exp(r*T1) + h*Ofin - payoff), h) for h in np.linspace(0,1.4,29))
cvar_stat_O, h_opt = best

# 4) delta-vega, rebalancement a chaque pas (+ variante avec bande de non-trading)
def dv_hedge(bandS, bandO):
    cash = np.full(m, premium); posS = np.zeros(m); posO = np.zeros(m)
    for k in range(n):
        tauL = max(T1-times[k], 0.01); tauO = T2-times[k]
        PS, Pv = greeks(Pinterp, S[:,k], v[:,k], tauL); OS, Ov = greeks(Ointerp, S[:,k], v[:,k], tauO)
        tO = Pv/Ov; tS = PS - tO*OS; Ok = Ointerp(np.c_[S[:,k], v[:,k], np.full(m, tauO)])
        trO = np.where(np.abs(tO-posO) > bandO, tO-posO, 0.); cash -= trO*Ok + cost*np.abs(trO)*Ok; posO += trO
        trS = np.where(np.abs(tS-posS) > bandS, tS-posS, 0.); cash -= trS*S[:,k] + cost*np.abs(trS)*S[:,k]; posS += trS
        cash *= np.exp(r*dt)
    return cvar(cash + posS*S[:,-1] + posO*Ofin - payoff)
cvar_dv      = dv_hedge(0.0, 0.0)
cvar_dv_band = dv_hedge(0.08, 0.10)

print(f"{'ne rien faire':38s} {cvar_none:7.2f}")
print(f"{'statique sous-jacent seul':38s} {cvar_stat_S:7.2f}")
print(f"{'delta-vega continu':38s} {cvar_dv:7.2f}")
print(f"{'delta-vega avec bande (cost-aware)':38s} {cvar_dv_band:7.2f}")
print(f"{'statique 1 option (h=%.2f)'%h_opt:38s} {cvar_stat_O:7.2f}")
print(f"{'reseau 2 instruments (notebook 12)':38s} {2.45:7.2f}")


### Lecture de la brique A

L'ordre est révélateur :

| Stratégie | CVaR 95% |
|---|---|
| Ne rien faire | ~35.0 |
| Statique, sous-jacent seul | ~16.1 |
| Delta-vega continu | ~6.54 |
| Delta-vega avec bande (cost-aware) | ~5.82 |
| **Statique, 1 option tenue** | **~3.90** |
| Réseau 2 instruments (nb 12) | ~2.45 |

Deux enseignements. Premièrement, **le gros du gain vient d'ajouter une option**, et il est capturé même sans rien faire de dynamique (35 → 3.90) : c'est l'effet de complétion du marché, et comme le passif et la couverture sont deux calls de même strike, ils co-bougent, donc un simple calendar spread statique absorbe l'essentiel. Deuxièmement, **toutes les versions du hedge grecque classique sont dominées** par ce simple hedge statique, parce qu'elles sur-tradent sous coût. La vraie valeur ajoutée du réseau n'est donc pas 6.54 → 2.45, c'est **3.90 → 2.45** : un raffinement dynamique d'environ 37%, réel mais modeste. Avec des calls de même strike, **le problème est trop facile pour que le dynamique compte vraiment**.


## Brique B — un passif que rien de statique ne réplique : l'option à barrière

Pour que le deep hedging brille, il faut un passif dont le risque est **path-dependent** et dont les grecques sont pathologiques. Le cas d'école est le **call up-and-out** : un call de strike $K=100$ qui est **désactivé** (payoff nul) si le prix touche une barrière $B=130$ à un moment de la vie de l'option.

$$\text{payoff} = (S_{T_1}-K)^+ \cdot \mathbb{1}\{\max_{t\le T_1} S_t < B\}.$$

Pourquoi c'est dur : quand $S$ monte vers la barrière, la valeur de l'option **s'effondre** (elle va bientôt être désactivée), donc son delta **change de signe** et son gamma explose. Un call vanille tenu statiquement fait exactement le contraire de ce qu'il faut près de la barrière (il est long l'upside, alors que le passif perd tout son upside). Il faut **réduire dynamiquement** la couverture en approchant $B$. Regardons d'abord le prix et le delta (formule fermée Black-Scholes de Reiner-Rubinstein).


In [ ]:
Bar = 130.0; sig_imp = 0.194     # barriere, et vol implicite ATM (pour la formule BS)

def bs_uo_call(S, X, H, T, r, sig):
    """Call up-and-out (X<H), monitoring continu : c_uo = A - B + C - D."""
    S = np.asarray(S, float); srt = sig*np.sqrt(T); m_ = (r - 0.5*sig**2)/sig**2
    x1 = np.log(S/X)/srt + (1+m_)*srt; x2 = np.log(S/H)/srt + (1+m_)*srt
    y1 = np.log(H**2/(S*X))/srt + (1+m_)*srt; y2 = np.log(H/S)/srt + (1+m_)*srt
    A = S*norm.cdf(x1) - X*np.exp(-r*T)*norm.cdf(x1-srt)
    B = S*norm.cdf(x2) - X*np.exp(-r*T)*norm.cdf(x2-srt)
    C = S*(H/S)**(2*(m_+1))*norm.cdf(-y1) - X*np.exp(-r*T)*(H/S)**(2*m_)*norm.cdf(-y1+srt)
    D = S*(H/S)**(2*(m_+1))*norm.cdf(-y2) - X*np.exp(-r*T)*(H/S)**(2*m_)*norm.cdf(-y2+srt)
    return np.where(S >= H, 0.0, A - B + C - D)
def bs_uo_delta(S, X, H, T, r, sig, h=0.5):
    return (bs_uo_call(S+h, X, H, T, r, sig) - bs_uo_call(S-h, X, H, T, r, sig))/(2*h)

Sgrid = np.linspace(70, 129.9, 200)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(Sgrid, bs_uo_call(Sgrid, K, Bar, 0.5, r, sig_imp), 'b', lw=2)
ax1.axvline(Bar, color='r', ls='--', label='barriere B=130'); ax1.axvline(K, color='k', lw=0.5)
ax1.set_title("Prix du call up-and-out (tau=0.5)"); ax1.set_xlabel("S"); ax1.legend()
ax2.plot(Sgrid, bs_uo_delta(Sgrid, K, Bar, 0.5, r, sig_imp), 'purple', lw=2)
ax2.axhline(0, color='k', lw=0.5); ax2.axvline(Bar, color='r', ls='--')
ax2.set_title("Delta : change de signe pres de la barriere"); ax2.set_xlabel("S")
fig.tight_layout(); plt.show()


On voit le prix culminer vers $S=110$ puis s'effondrer vers la barrière, et le delta passer de $+0.27$ à la monnaie à $-0.48$ près du seuil. Aucune position statique ne suit ça.


### L'échelle des stratégies classiques sur la barrière

Prime du passif : prix Heston de l'up-and-out, calculé par Monte Carlo risque-neutre (pas de forme fermée sous Heston). Puis on compare : ne rien faire, le delta hedge classique (delta BS-barrière, on liquide une fois la barrière franchie), et le vanille statique.


In [ ]:
# prime up-and-out sous Heston (MC risque-neutre)
Sq, _ = sim_heston(200_000, r, 1)
knq = (Sq.max(axis=1) >= Bar)
prem_bar = float(np.exp(-r*T1)*np.where(~knq, np.maximum(Sq[:,-1]-K, 0.), 0.).mean())
print(f"prime up-and-out (Heston, B=130) = {prem_bar:.3f}   proba de knock-out = {knq.mean():.1%}")

# trajectoires reelles + statut 'vivant' (1 tant que la barriere n'a jamais ete touchee)
alive = np.cumprod((S < Bar).astype(float), axis=1)
payoff_bar = np.where(alive[:,-1] > 0, np.maximum(S[:,-1]-K, 0.), 0.)

cvar_bar_none = cvar(prem_bar*np.exp(r*T1) - payoff_bar)

# delta hedge classique BS-barriere : cible = delta * vivant (on liquide si knock)
cash = np.full(m, prem_bar); posS = np.zeros(m)
for k in range(n):
    tau = max(T1-times[k], 1e-3)
    tgt = bs_uo_delta(S[:,k], K, Bar, tau, r, sig_imp)*alive[:,k]
    tr = tgt - posS; cash -= tr*S[:,k] + cost*np.abs(tr)*S[:,k]; posS = tgt; cash *= np.exp(r*dt)
cvar_bar_delta = cvar(cash + posS*S[:,-1] - payoff_bar)

# statique : 1 vanille (2 ans) tenu, h optimise
best = min((cvar((prem_bar-h*O0-cost*abs(h)*O0)*np.exp(r*T1) + h*Ofin - payoff_bar), h) for h in np.linspace(-0.2,1.2,29))
cvar_bar_statO, h_bar = best

print(f"{'ne rien faire':34s} {cvar_bar_none:7.2f}")
print(f"{'delta hedge BS-barriere (dynamique)':34s} {cvar_bar_delta:7.2f}")
print(f"{'statique 1 vanille (h=%.2f)'%h_bar:34s} {cvar_bar_statO:7.2f}")


### Ce que ça montre

| Stratégie (barrière) | CVaR 95% |
|---|---|
| Ne rien faire | ~20.7 |
| **Delta hedge classique BS-barrière** | **~26.5** |
| Statique 1 vanille | ~6.44 |

Le delta hedge classique de la barrière est **pire que ne rien faire** : près du seuil il exige des positions short énormes, et avec un rééquilibrage discret plus le gap de désactivation, il perd gros dans la queue. C'est le résultat classique qui fait des barrières le cas d'école du deep hedging. Le vanille statique (6.44) est bien meilleur mais laisse énormément sur la table, car il ne peut pas *réduire* la couverture en approchant la barrière. Place au réseau.


### Le deep hedger de barrière (résiduel + Monte Carlo frais)

Deux essais avaient échoué (CVaR ~10, pire que le statique). Diagnostic : sur un lot de trajectoires **figé**, le réseau surapprend la queue (les ~10% de chemins près de la barrière), et généralise mal. Deux corrections :

1. **Hedging résiduel + init à zéro.** Le réseau apprend une correction $\Delta$ autour d'une ancre statique ($H_0=0.9$ vanille), et sa dernière couche est initialisée à zéro, donc au départ la politique *est* le hedge statique (CVaR ~6.44) : la descente ne peut qu'améliorer.
2. **Monte Carlo frais.** On resimule un lot Heston **neuf à chaque époque** au lieu d'un jeu figé. Il n'y a plus rien à mémoriser, donc plus de surapprentissage. On imprime CVaR train *et* test pour le vérifier.

État du réseau : les 6 features du nb12 plus la **distance à la barrière** $(B-S)/S_0$ et le **statut vivant** (0/1).

In [ ]:
import torch
torch.manual_seed(0)

def price_along(S, v, interp, Tmat):
    m, n1 = S.shape; tt = np.linspace(0, T1, n1); O = np.empty_like(S)
    for k in range(n1):
        O[:,k] = interp(np.c_[S[:,k], v[:,k], np.full(m, max(Tmat-tt[k], 1e-3))])
    return O

def make_batch(m, seed):
    """Un lot Heston FRAIS + prix option + statut vivant, en tenseurs (constantes)."""
    Sb, vb = sim_heston(m, mu, seed)
    Ob = price_along(Sb, vb, Ointerp, T2)
    Ab = np.cumprod((Sb < Bar).astype(np.float32), axis=1)
    return (torch.tensor(Sb, dtype=torch.float32), torch.tensor(vb, dtype=torch.float32),
            torch.tensor(Ob, dtype=torch.float32), torch.tensor(Ab, dtype=torch.float32))

# jeu de test FIXE (jamais vu a l'entrainement puisque les lots train sont regeneres)
S_te2, v_te2 = sim_heston(80_000, mu, 200)
Ste = torch.tensor(S_te2, dtype=torch.float32); vte = torch.tensor(v_te2, dtype=torch.float32)
Ote = torch.tensor(price_along(S_te2, v_te2, Ointerp, T2), dtype=torch.float32)
Ate = torch.tensor(np.cumprod((S_te2 < Bar).astype(np.float32), axis=1), dtype=torch.float32)

def cvar_torch(L, a=0.95):
    """CVaR empirique direct : moyenne des pertes au-dela du quantile a. Cible la QUEUE."""
    var = torch.quantile(L, a)
    return L[L >= var].mean()

H0 = 0.90   # ancre statique (vanille)
class BarrierNet(torch.nn.Module):
    def __init__(self, h=64):
        super().__init__()
        self.net = torch.nn.Sequential(torch.nn.Linear(8, h), torch.nn.ReLU(),
                                       torch.nn.Linear(h, h), torch.nn.ReLU(),
                                       torch.nn.Linear(h, 2))
        torch.nn.init.zeros_(self.net[-1].weight); torch.nn.init.zeros_(self.net[-1].bias)
    def forward(self, x): return self.net(x)

def positions(net, feat):
    a = net(feat)
    return a[:,0], H0 + a[:,1]      # (sous-jacent autour de 0, option autour de H0)

def hedge_pnl(net, S, v, O, A):
    m = S.shape[0]; cash = torch.full((m,), prem_bar); posS = torch.zeros(m); posO = torch.zeros(m)
    for k in range(n):
        tau = float(T1 - k*dt)
        feat = torch.stack([torch.log(S[:,k]/K), torch.full((m,), tau), posS, posO,
                            v[:,k], O[:,k]/S0, (Bar - S[:,k])/S0, A[:,k]], dim=1)
        tgS, tgO = positions(net, feat); trS = tgS-posS; trO = tgO-posO
        cash = cash - trS*S[:,k] - cost*torch.abs(trS)*S[:,k]
        cash = cash - trO*O[:,k] - cost*torch.abs(trO)*O[:,k]
        cash = cash*np.exp(r*dt); posS = tgS; posO = tgO
    payoff = torch.clamp(S[:,-1]-K, min=0.0)*A[:,-1]
    return cash + posS*S[:,-1] + posO*O[:,-1] - payoff

In [ ]:
net = BarrierNet()
opt = torch.optim.Adam(net.parameters(), lr=5e-4)
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=250, gamma=0.4)

# CONTROLE : le reseau a l'init = ancre statique (0.9 vanille) => doit donner ~6.4
net.eval()
with torch.no_grad():
    c0 = cvar(hedge_pnl(net, Ste, vte, Ote, Ate).numpy())
net.train()
print(f"CVaR test a l'init = {c0:.3f}   (doit valoir ~ statique {cvar_bar_statO:.2f})")

batch, epochs = 20_000, 700
for ep in range(epochs):
    Sb, vb, Ob, Ab = make_batch(batch, seed=1000+ep)          # lot Heston FRAIS
    loss = cvar_torch(-hedge_pnl(net, Sb, vb, Ob, Ab))         # CVaR empirique direct
    opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    if (ep+1) % 50 == 0:
        net.eval()
        with torch.no_grad():
            cte = cvar(hedge_pnl(net, Ste, vte, Ote, Ate).numpy())
        net.train()
        print(f"epoch {ep+1:4d}   CVaR train = {loss.item():6.3f}   CVaR test = {cte:6.3f}")

In [ ]:
net.eval()
with torch.no_grad():
    pnl_barnet = hedge_pnl(net, Ste, vte, Ote, Ate).numpy()
print(f"{'ne rien faire':34s} {cvar_bar_none:7.2f}")
print(f"{'delta hedge BS-barriere':34s} {cvar_bar_delta:7.2f}")
print(f"{'statique 1 vanille':34s} {cvar_bar_statO:7.2f}")
print(f"{'reseau residuel (barriere)':34s} {cvar(pnl_barnet):7.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
labels = ['ne rien\nfaire','delta BS\nbarriere','statique\n1 vanille','reseau\n2 instr.']
vals = [cvar_bar_none, cvar_bar_delta, cvar_bar_statO, cvar(pnl_barnet)]
ax.bar(labels, vals, color=['gray','tab:red','tab:orange','tab:green'])
for i,val in enumerate(vals): ax.text(i, val+0.3, f"{val:.2f}", ha='center')
ax.set_title("Couverture d'un call up-and-out sous Heston (CVaR 95%, plus bas = mieux)")
ax.set_ylabel("CVaR"); fig.tight_layout(); plt.show()


## Ce qu'il faut retenir

- **Sois honnête sur d'où vient un chiffre.** Le 2.45 du notebook 12 venait surtout d'*ajouter* une option (complétion du marché), pas du trading dynamique : un hedge statique donnait déjà 3.90, et le benchmark delta-vega classique sur-tradait. Savoir décomposer son propre résultat vaut plus qu'un joli chiffre.
- **Le deep hedging brille quand le problème est vraiment dur.** Sur un call de même strike, le statique suffit presque. Sur une **barrière**, le delta hedge classique est *pire que ne rien faire* (positions explosives près du seuil, gap de désactivation), le vanille statique plafonne, et le réseau apprend à *réduire* la couverture en approchant la barrière, ce qu'aucune grecque ne lui dit de faire.
- **La recette qui rend le réseau meilleur** : il optimise directement le CVaR sous coûts et discrétisation, avec l'information de path-dependence (distance à la barrière, statut vivant) dans son état. C'est une politique, pas une formule.
